In [1]:
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Tools.Visualization import qs_help

In [2]:
import warnings
warnings.filterwarnings('ignore')
import logging

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

以下示例要求有可以访问的 Wind 量化研究数据库，并已在本地配置文件中设置连接信息。示例表名和字段以 QSExt 自带的 `WindDB2Info.xlsx` 为准，实际可用数据取决于数据库的表结构与权限。

# WindDB2

`WindDB2`（聚源因子库）是基于关系数据库（PostgreSQL / MySQL / SQL Server / Oracle）构建的因子库，通过 SQL 查询来完成因子的数据访问。

## 存储映射

WindDB2 将关系数据库映射为因子模型：

| 逻辑层 | 存储层 |
|--------|--------|
| 因子库（WindDB2） | 整个数据库 |
| 因子表（FactorTable） | 数据库中的一张表 |
| 因子（Factor） | 数据库表中的一个字段 |

本质上，WindDB2 将数据库表的多列数据重组为三维因子数据（因子 × 时点 × 证券代码）。因子表的类型（`TableType`）决定了这个重组的具体方式。

```mermaid
graph TD
    subgraph 逻辑层
        A[因子库<br/>WindDB2]
        B1[因子表<br/>FactorTable]
        B2[因子表]
        A --> B1
        A --> B2
        F1[因子1]
        F2[因子2]
        B1 --> F1
        B1 --> F2
    end

    subgraph 存储层
        C[关系数据库]
        D1[数据库表]
        D2[数据库表]
        C --> D1
        C --> D2
        E2[字段2]
        E1[字段1]
        D2 --> E1
        D2 --> E2
    end

    A -.-> C
    B1 -.-> D2
    F1 -.-> E1

    style A fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style B1 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style B2 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style F1 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style F2 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style C fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style D1 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style D2 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style E1 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style E2 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
```

## 配置与连接

`WindDB2` 默认查找 `~/QuantStudioConfig/WindDB2Config.json`。连接凭据保存在本地配置文件中。参数优先级为：显式 `args` > 配置文件 > 模型默认值。

默认库信息文件为 QSExt 自带的 `Resource/WindDB2Info.xlsx`；自定义 `DBInfoFile` 应包含 `TableInfo`、`FactorInfo` 工作表。元数据加载时会适配历史字段类型及重复记录，不修改原始 Excel。

`TableNames` 会按实际数据库中存在的表筛选，因此应先连接数据库。

In [3]:
from QSExt.Factor.WindDB2 import WindDB2

FDB = WindDB2().connect()
print(qs_help(FDB))

类型: WindDB2
模块: QSExt.Factor.WindDB2
QS 对象类型: 因子库
QS 对象名称: WindDB2
QSID: 408c6b36f6894d3fd9400b3752adeee3a92863502c0f2a88a8f3aeee788cc9ee
参数集:
    * Name(名称): <class 'str'>, 默认值 'WindDB2', 当前取值: 'WindDB2'
    * DBType(数据库类型): typing.Literal['MySQL', 'SQL Server', 'Oracle', 'PostgreSQL'], 默认值 'MySQL', 当前取值: 'PostgreSQL'
    * DBName(数据库名): <class 'str'>, 默认值 'Scorpion', 当前取值: 'WindDB'
    * IPAddr(IP地址): <class 'str'>, 默认值 '127.0.0.1', 当前取值: 'localhost'
    * Port(端口): <class 'int'>, 默认值 3306, 当前取值: 5432
    * User(用户名): <class 'str'>, 默认值 'root', 当前取值: 'postgres'
    * TablePrefix(表名前缀): <class 'str'>, 默认值 '', 当前取值: ''
    * CharSet(字符集): typing.Literal['utf8', 'utf8mb4', 'gbk', 'gb2312', 'gb18030', 'cp936', 'big5'], 默认值 'utf8', 当前取值: 'utf8'
    * Connector(连接器): typing.Literal['default', 'cx_Oracle', 'pymssql', 'mysql.connector', 'pymysql', 'psycopg2', 'pyodbc'], 默认值 'default', 当前取值: 'default'
    * ConnRetryNum(连接重试次数): <class 'int'>, 默认值 3, 当前取值: 3
    * ConnIntervalSeconds(连接重试间隔):

In [4]:
# 获取可用的因子表列表
print(FDB.TableNames[:5])

['香港交易所交易日历', '中国A股交易日历', '中国A股基本资料', '中国期货基本资料', '中国债券市场交易日']


## 因子表类型概览

WindDB2 将关系数据库的行列数据重组为 `Panel(items=因子, major_axis=时点, minor_axis=ID)`。默认按库信息文件中的映射选择表类型，也可通过 `getTable(args={"TableType": ...})` 指定。

| 类型标识 | 数据组织方式 | 典型示例 |
|----------|--------------|----------|
| `WideTable` | 每行对应一个 ID 和时点，各字段为因子 | 中国A股日行情 |
| `FeatureTable` | 证券属性，无时点维度或读取最新值 | 中国A股基本资料 |
| `TimeSeriesTable` | 无 ID 维度，各 ID 共享同一时序 | 中国共同基金股票风格分类门限值 |
| `NarrowTable` | 每行记录 ID、时点、因子名和因子值 | 需指定匹配的底层字段 |
| `MappingTable` | 在起始和截止时点之间填充属性 | 中国A股Wind行业分类 |
| `ConstituentTable` | 按类别展开成份股 0/1 标记 | 中国A股指数成份股 |
| `FinancialTable` | 按公告日期及报告期读取、转换财务数据 | 中国A股资产负债表 |
| `AnalystConsensusTable` | 分析师一致预期 | 中国A股盈利预测汇总 |
| `AnalystEstDetailTable` | 分析师盈利预测明细 | 中国A股盈利预测明细 |
| `AnalystRatingDetailTable` | 分析师评级明细 | 中国A股投资评级明细 |

因子表参数优先级为：`getTable(args=...)` > 表元数据 `DefaultArgs` > 库参数 `FTArgs` > 默认值。推荐英文参数名；同一层同时给出中英文键时以英文键为准。默认 `DTFmt` 和 `DateFmt` 均为 `%Y%m%d`。

# 时点与 ID 获取

WindDB2 提供了一系列方法直接查询数据库获取时点序列和证券代码序列，无需先获取因子表。

## 交易日序列

In [5]:
# 获取指定交易所和日期范围的交易日
DTs = FDB.getTradeDay(start_date=dt.datetime(2022, 1, 1), end_date=dt.datetime(2022, 1, 5))
print(DTs)

[Timestamp('2022-01-04 00:00:00'), Timestamp('2022-01-05 00:00:00')]


## 证券代码序列

| 方法 | 用途 | 特有参数 |
|------|------|----------|
| `getStockID` | A 股或指定指数的成份股 | `index_id="全体A股"` 或 Wind 指数代码 |
| `getMutualFundID` | 公募基金 | `exchange`、`start_date` |
| `getFutureID` | 期货合约 | `future_code`、`contract_type`、`include_simulation` |
| `getFutureCode` | 期货品种代码 | `exchange`、`include_simulation` |
| `getOptionID` | 期权合约 | `option_code="510050OP"` |

`date=None` 表示当前日期。`is_current=True` 按指定日的在市条件筛选，`False` 放宽退市或到期条件；具体边界由各证券类型决定。基金还支持 `start_date`：`is_current=True` 查询区间内始终存续的基金，`False` 查询区间内曾存续的基金。

这些方法保留 Wind 的查询口径和代码格式，交易所筛选值也应使用 Wind 数据库中的编码。

In [6]:
# 获取全体A股（含已退市）
IDs = FDB.getStockID(is_current=False)
print(IDs[:5])

['000001.SZ', '000002.SZ', '000003.SZ', '000004.SZ', '000005.SZ']


In [7]:
# 获取全体公募基金（含已清盘）
IDs = FDB.getMutualFundID(is_current=False)
print(IDs[:5])

['000001.OF', '000003.OF', '000004.OF', '000005.OF', '000006.OF']


In [8]:
IDs = FDB.getFutureID(future_code="IF", is_current=False)
print(IDs[:5])

['IF1005.CFE', 'IF1006.CFE', 'IF1007.CFE', 'IF1008.CFE', 'IF1009.CFE']


In [9]:
IDs = FDB.getFutureCode(is_current=False)
print(IDs[:5])

['A', 'AD', 'ADBC10', 'ADBC2_P', 'ADBC5']


In [10]:
IDs = FDB.getOptionID(option_code="510050OP", is_current=False)
print(IDs[:5])

['10000001.SH', '10000002.SH', '10000003.SH', '10000004.SH', '10000005.SH']


## 读取数据

因子表的 `readData` 从数据库执行 SQL 查询并重组为 `Panel` 返回。

In [11]:
# 因子表读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["510050.OF", "512910.OF"]

FT = FDB.getTable("中国ETF申购赎回清单", args={"LookBack": 0})
Data = FT.readData(factor_names=["最小申购赎回单位资产净值", "现金差额"], ids=IDs, dts=DTs)
print("因子表数据 (Panel):")
print(Data)

因子表数据 (Panel):
<class 'QuantStudio.Core.QSObject.Panel'>
Dimensions: 2 (items) x 5 (major_axis) x 2 (minor_axis)
Items axis: 最小申购赎回单位资产净值 to 现金差额
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-05 00:00:00
Minor_axis axis: 510050.OF to 512910.OF


# 因子表

In [13]:
# 获取因子表对象 — 以中国ETF申购赎回清单（WideTable）为例
FT = FDB.getTable("中国ETF申购赎回清单", args={"LookBack": 0})
print(qs_help(FT))

类型: _WideTable
模块: QSExt.Factor.WindDB2
QS 对象类型: 计算节点-因子表
QS 对象名称: 中国ETF申购赎回清单
QSID: 698f8d675b33e9d371e11f3a7282d16181cec8c7e44567dbc50aab55a11bc002
参数集:
    * Name(因子表名称): <class 'str'>, 默认值 'FactorTable', 当前取值: '中国ETF申购赎回清单'
    * TableType(因子表类型): <class 'str'>, 默认值 'SQL_Table', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'SQL_Table'
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: '交易日期'
    * AdditionalCondition(附加条件): <class 'dict'>, 默认值 {}, 当前取值: {}
    * LookBack(回溯天数): typing.Union[int, float], 默认值 0, 缺失填充回溯的天数, 0 表示不回溯填充, 当前取值: 0
    * PublDTField(公告时点字段): typing.Optional[str], 默认值 None, 用作公告时点的字段名, 默认值 None 表示内部自动判断, 如果非 None, 表示考虑数据的公布时点, 即某个时点所能获取的数据必须保证其在公告时点和截止时点之后, 当前取值: None
    * MultiMapping(多重映射): <class 'bool'>, 默认值 False, 是否为高维数据, 即时点和 ID 两个维度无法唯一索引单个数据, 默认形成的数据在单个时点单个 ID 处以 list 形式表达, 当前取值: False
    * Operator(算子): typing.Optional[typing.Callable], 默认值 None, 对于单个时点单个 ID 处的数据 apply 的函数 f(x), 其中 x 为 Series, 默认值 

In [14]:
# 获取因子列表
print(FT.FactorNames)

['Wind代码', '交易日期', '现金差额', '最小申购赎回单位资产净值', '预估现金部分', '现金替代比例上限', '最小申购赎回单位', '申购赎回允许情况 ', '标的指数成分股数量']


# WideTable 详解

WideTable 是最常用的因子表类型，每行数据唯一对应一个（ID, 时点）组合，每个字段（除 ID 和时点字段外）是一个因子。

![Wide_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Wide_Table_Demo.png)

WideTable 的核心特性：

## 缺失填充（LookBack）

数据库中的数据常常是稀疏的（如非交易日无行情数据），`LookBack` 参数控制如何回填空缺。

In [ ]:
# LookBack=0：不填充，非交易日为 NaN
FT = FDB.getTable("中国A股日行情", args={"LookBack": 0})

DTs = [dt.datetime(2023, 12, 29) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ"]
Data = FT.readData(factor_names=["收盘价(元)", "开盘价(元)"], ids=IDs, dts=DTs).iloc[:, :, 0]
print("LookBack=0 :")
print(Data)

In [ ]:
# LookBack>0：向前回溯填充缺失值，inf 表示无限回溯
FT = FDB.getTable("中国A股日行情", args={"LookBack": 2})
Data = FT.readData(factor_names=["收盘价(元)", "开盘价(元)"], ids=IDs, dts=DTs).iloc[:, :, 0]
print("LookBack=2 :")
print(Data)

## 公告时点（PublDTField）

对于财务数据，财报有报告期和公告日期之分。在公告日之前，即使报告期已过，也不应看到该数据（避免未来信息）。`PublDTField` 参数指定公告日期字段，确保数据只在公告后才能被读取到。

![Wide_Table_AnnDT_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Wide_Table_AnnDT_Demo.png)

In [ ]:
DTs = [dt.datetime(2023, 6, 30), dt.datetime(2023, 9, 30), dt.datetime(2023, 10, 1)]
IDs = ["000001.SZ"]

# 固定报表类型，比较是否按公告日期约束数据可见性。
for PublDTField in (None, "公告日期"):
    FT = FDB.getTable("中国A股资产负债表", args={
        "TableType": "WideTable", "LookBack": np.inf,
        "PublDTField": PublDTField,
        "AdditionalCondition": {"报表类型": "408001000"}
    })
    Data = FT.readData(factor_names=["公告日期", "资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
    print("PublDTField =", PublDTField)
    print(Data)

## 多重映射（MultiMapping）

当一个（ID, 时点）对应多个数据行时（如同一报告期有多份修正财报），启用 `MultiMapping=True` 可将多值聚合为 list。可通过 `Operator` 参数指定聚合函数（如取均值）。

![Wide_Table_MultiMapping_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Wide_Table_MultiMapping_Demo.png)

In [ ]:
DTs = [dt.datetime(2023, 9, 30), dt.datetime(2023, 12, 31)]
IDs = ["000001.SZ"]

# 同一报告期不同报表类型可能产生多行；先查看列表，再演示聚合。
for Operator in (None, lambda x: x.mean()):
    FT = FDB.getTable("中国A股资产负债表", args={
        "TableType": "WideTable", "LookBack": 0, "PublDTField": None,
        "MultiMapping": True, "Operator": Operator, "OperatorDataType": "double"
    })
    Data = FT.readData(factor_names=["资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
    print(Data)

此处仅演示多值处理接口。不同报表类型的均值通常没有财务分析意义；实际研究应先明确报表口径并筛选。

# 其他常用因子表类型

## FeatureTable

继承自 WideTable，适用于**没有时点维度**的数据（如证券基本属性），或只取最新值的场景。默认 `LookBack=inf`，所有请求的时点都返回相同的数据。

![Feature_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Feature_Table_Demo.png)

In [ ]:
# FeatureTable：所有时点数据一样
FT = FDB.getTable("中国A股基本资料")
DTs = [dt.datetime(2025, 1, 1), dt.datetime(2025, 1, 2)]
Data = FT.readData(factor_names=["证券简称", "上市日期"], ids=["000001.SZ"], dts=DTs).iloc[:, :, 0]
print(Data)

## TimeSeriesTable

无 ID 字段，只有时点字段的时序数据表。每个时点只有一行数据，所有 ID 共享同一套值。可以理解为 ID 字段为单一值的 WideTable。默认 `LookBack=inf`。

典型场景：宏观经济指标（如无风险利率、GDP 增速），对所有证券都相同。

![TimeSeries_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/TimeSeries_Table_Demo.png)

## NarrowTable

高瘦表（或叫"长表"），每行是一个四元组：**(ID, 时点, 因子名称, 因子值)**。通过 `FactorNameField` 指定因子名称字段，`FactorValueField` 指定因子值字段，系统将其 pivot 为宽表形式的因子数据。默认 `MultiMapping=True`。

![Narrow_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Narrow_Table_Demo.png)

## MappingTable

在**起始时点到截止时点之间**填充相同的值。典型应用于行业分类数据：某只股票在某段时间内属于某个行业，该时间段内所有时点都填充相同的行业代码。

核心参数：
| 参数 | 说明 |
|------|------|
| `DTField` | 起始时点字段 |
| `EndDTField` | 截止时点字段 |
| `EndDTIncluded` | 截止时点处是否填充数据（默认 `True`） |

![Mapping_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Mapping_Table_Demo.png)

In [ ]:
FT = FDB.getTable("中国A股Wind行业分类")
DTs = [dt.datetime(2018, 2, 1), dt.datetime(2019, 12, 3)]
Data = FT.readData(factor_names=["Wind行业代码"], ids=["600136.SH"], dts=DTs).iloc[:, :, 0]
print(Data)

## ConstituentTable

成份股数据，因子值是一个 0/1 哑变量，1 表示该证券属于对应指数成份。`GroupField` 参数指定类别字段，其每个取值展开为一个因子。

核心参数：
| 参数 | 说明 |
|------|------|
| `DTField` | 入选日期字段 |
| `EndDTField` | 剔除日期字段 |
| `GroupField` | 类别字段（如 Wind 指数代码），每个取值成为一个因子 |
| `EndDTIncluded` | 截止时点是否仍属于成份（默认 `False`） |
| `GroupMapping` | 类别值的名称映射（SQL 或 dict） |

![Constituent_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Constituent_Table_Demo.png)

In [ ]:
FT = FDB.getTable("中国A股指数成份股")
DTs = [dt.datetime(2018, 2, 1), dt.datetime(2019, 12, 3)]
Data = FT.readData(factor_names=["000016.SH", "000300.SH"], ids=["000001.SZ"], dts=DTs).iloc[:, :, 0]
print(Data)

## FinancialTable

最复杂的因子表类型，专门处理财务报表数据。财务数据的关键特征是每条记录有**报告期**和**公告日期**，需要根据当前时点判断应该使用哪个报告期的数据。

核心参数：

| 参数 | 取值 | 说明 |
|------|------|------|
| `CalcType` | `"最新"` / `"单季度"` / `"TTM"` | 财务数据转换成因子值的方式 |
| `ReportDate` | `"所有"` / `"定期报告"` / `"年报"` / `"中报"` / `"一季报"` / `"三季报"` | 使用的报告期类型 |
| `PublDTField` | `str \| None` | 公告日期字段，用于控制数据在公告后才可用 |
| `YearLookBack` | `int` | 回溯年数：取 n 年前同报告期的数据 |
| `PeriodLookBack` | `int` | 回溯期数：取 n 期前同报告期的数据 |

**CalcType 详解：**

- **最新**：当前时点能看到的最新报告期数据。例如 2010-08-20 时某公司已披露 2010 年中报，则用中报数据
- **单季度**：当前时点能看到的单季度数据。例如 2010-08-20 已披露中报 → 净利润(中报) - 净利润(一季报) = Q2 净利润
- **TTM**：当前时点能看到的滚动四季度数据。例如 2010-08-20 已披露中报 → 净利润(中报) + 净利润(去年年报) - 净利润(去年中报)

![Financial_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Financial_Table_Demo.png)

In [ ]:
# FinancialTable：CalcType="最新"，ReportDate="年报"
FT = FDB.getTable("中国A股资产负债表", args={
    "ReportDate": "年报",
    "CalcType": "最新"
})
DTs = [dt.datetime(2018, 2, 1), dt.datetime(2019, 12, 3)]
Data = FT.readData(factor_names=["资产总计"], ids=["000001.SZ"], dts=DTs).iloc[:, :, 0]
print(Data)

### Wind 财务口径

`ReportType` 默认保留 `408001000`、`408004000`、`408005000`。含报表类型字段时，`408004000` 使用本表公告日期，其余类型优先采用实际披露日期，缺少实际披露日期时回退到本表公告日期。财务读取只使用当前时点已经公告的数据。

`YearLookBack` 和 `PeriodLookBack` 同时非零时优先按年数回溯。`ExprFactor`、`NoticeFactor` 分别指定业绩快报、业绩预告表的补充字段；预告值按万元转元乘以 `10000`，使用前需确认与目标财务因子口径一致。

`TTM`、`单季度` 适用于累计流量指标；资产总计等存量指标通常读取最新值。

In [ ]:
FT = FDB.getTable("中国A股利润表", args={"CalcType": "TTM"})
DTs = [dt.datetime(2024, 4, 29), dt.datetime(2024, 4, 30)]
Data = FT.readData(factor_names=["营业收入"], ids=["000001.SZ"], dts=DTs).iloc[:, :, 0]
print(Data)

# 分析师因子表

## AnalystConsensusTable

汇总表通过 `CalcType` 选择 `FY0`、`FY1`、`FY2` 或 `Fwd12M`；`LookBack` 控制预测数据回溯天数，`Period` 使用 Wind 周期代码，与明细表中表示天数的 `Period` 不同。

FY 年度在上一年年报实际公告当日切换；缺少公告记录时按尚未披露处理。`Fwd12M` 按年度剩余天数对预测值加权。

In [ ]:
FT = FDB.getTable("中国A股盈利预测汇总", args={
    "CalcType": "FY1", "Period": "263001000", "LookBack": 180
})
print(FT.FactorNames[:5])

## AnalystRatingDetailTable 与 AnalystEstDetailTable

明细表需要传入业务 `Operator`，默认算子返回缺失值。

| 参数 | 说明 |
|------|------|
| `Period` | 回溯窗口天数，窗口为 `(当前日 - Period, 当前日]` |
| `Deduplication` | 默认按研究机构保留窗口内最新日期的记录；同日多条记录仍可能保留 |
| `AdditionalFields` | 除当前因子外，算子还需要的字段 |
| `ModelArgs` | 传递给算子的业务参数 |
| `DataType` | 算子返回值类型：`double`、`string`、`object` |
| `ForwardYears` | 盈利预测明细的预测年度偏移列表，默认 `[0]` |

算子签名为 `operator(table, date, iid, x, args)`。评级表的 `x` 为 DataFrame，盈利预测明细的 `x` 为与 `ForwardYears` 对应的 DataFrame 列表。ID 列统一为 `QS_ID`，日期列为 `日期`，日期值仍为 `%Y%m%d` 字符串。

分析师数据按日计算，再恢复请求时点的顺序；同一天多个盘中时点得到相同的日级结果。预测原始数据及年报公告数据分别通过 `FactorCache` 缓存，支持按 ID 分配给进程。

In [ ]:
def mean_rating(table, date, iid, x, args):
    """计算窗口内各机构最新记录的平均标准评级。

    Args:
        table: 当前因子表。
        date: 当前计算日期。
        iid: 当前证券代码。
        x: 筛选后的评级记录。
        args: 业务参数。

    Returns:
        float: 平均标准评级，无有效记录时为 NaN。
    """
    return pd.to_numeric(x["本次标准评级"], errors="coerce").mean()

FT = FDB.getTable("中国A股投资评级明细", args={"Period": 180, "Operator": mean_rating})
Data = FT.readData(factor_names=["本次标准评级"], ids=["000001.SZ"], dts=[dt.datetime(2024, 1, 2)])
print(Data.iloc[0])

# 因子

通过 `FT.getFactor()` 获取的因子与其他因子库共享因子读取和计算图接口。下面重新指定日行情表，避免依赖前面示例中的 `FT`。

In [ ]:
FT = FDB.getTable("中国A股日行情", args={"LookBack": 0})
F = FT.getFactor("收盘价(元)")
DTs = [dt.datetime(2024, 1, 2), dt.datetime(2024, 1, 3)]
Data = F.readData(ids=["000001.SZ", "000002.SZ"], dts=DTs)
print(Data)

In [ ]:
FDB.disconnect()